In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [4]:
df = pd.read_csv(
    "../data/raw/loan_data.csv"
)
X = df.drop(
    columns=["loan_status"]
)

y = df["loan_status"]

In [5]:
X["loan_to_income"] = (
    X["loan_amnt"] /
    X["person_income"]
)

In [6]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [7]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [8]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

In [13]:
logistic_params = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "model__solver": [
        "liblinear",
        "lbfgs"
    ],

    "model__class_weight": [
        None,
        "balanced"
    ]
}

In [14]:
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_params,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [15]:
logistic_grid.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__class_weight': [None, 'balanced'], 'model__solver': ['liblinear', 'lbfgs']}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [16]:
logistic_grid.best_params_

{'model__C': 10, 'model__class_weight': None, 'model__solver': 'liblinear'}

In [17]:
logistic_grid.best_score_

np.float64(0.9536986830357144)

In [18]:
best_logistic = (
    logistic_grid.best_estimator_
)

In [19]:
knn_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            KNeighborsClassifier()
        )
    ]
)

In [20]:
knn_params = {
    "model__n_neighbors": [
        3,
        5,
        7,
        9,
        11,
        15,
        21
    ],

    "model__weights": [
        "uniform",
        "distance"
    ],

    "model__metric": [
        "euclidean",
        "manhattan"
    ]
}


In [21]:
knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_params,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [22]:
knn_grid.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 28 candidates, totalling 140 fits


,estimator,Pipeline(step...lassifier())])
,param_grid,"{'model__metric': ['euclidean', 'manhattan'], 'model__n_neighbors': [3, 5, ...], 'model__weights': ['uniform', 'distance']}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [23]:
best_knn = (
    knn_grid.best_estimator_
)

In [24]:
nb_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            GaussianNB()
        )
    ]
)

In [25]:
nb_params = {
    "model__var_smoothing": [
        1e-12,
        1e-11,
        1e-10,
        1e-9,
        1e-8,
        1e-7,
        1e-6,
        1e-5
    ]
}

In [26]:
nb_grid = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=nb_params,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [27]:
nb_grid.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,estimator,Pipeline(step...aussianNB())])
,param_grid,"{'model__var_smoothing': [1e-12, 1e-11, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [28]:
nb_grid.best_params_

{'model__var_smoothing': 1e-12}

In [29]:
nb_grid.best_score_

np.float64(0.9359082924107142)

In [30]:
best_nb = (
    nb_grid.best_estimator_
)

In [31]:
logistic_pred = (
    best_logistic.predict(X_test)
)

logistic_prob = (
    best_logistic.predict_proba(
        X_test
    )[:, 1]
)

In [32]:
knn_pred = (
    best_knn.predict(X_test)
)

knn_prob = (
    best_knn.predict_proba(
        X_test
    )[:, 1]
)

In [33]:
nb_pred = (
    best_nb.predict(X_test)
)

nb_prob = (
    best_nb.predict_proba(
        X_test
    )[:, 1]
)

In [34]:
tuned_results = pd.DataFrame({
    "Model": [
        "Tuned Logistic Regression",
        "Tuned KNN",
        "Tuned Gaussian Naive Bayes"
    ],

    "Accuracy": [
        accuracy_score(
            y_test,
            logistic_pred
        ),
        accuracy_score(
            y_test,
            knn_pred
        ),
        accuracy_score(
            y_test,
            nb_pred
        )
    ],

    "Precision": [
        precision_score(
            y_test,
            logistic_pred
        ),
        precision_score(
            y_test,
            knn_pred
        ),
        precision_score(
            y_test,
            nb_pred
        )
    ],

    "Recall": [
        recall_score(
            y_test,
            logistic_pred
        ),
        recall_score(
            y_test,
            knn_pred
        ),
        recall_score(
            y_test,
            nb_pred
        )
    ],

    "F1": [
        f1_score(
            y_test,
            logistic_pred
        ),
        f1_score(
            y_test,
            knn_pred
        ),
        f1_score(
            y_test,
            nb_pred
        )
    ],

    "ROC-AUC": [
        roc_auc_score(
            y_test,
            logistic_prob
        ),
        roc_auc_score(
            y_test,
            knn_prob
        ),
        roc_auc_score(
            y_test,
            nb_prob
        )
    ]
})


In [35]:
tuned_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Tuned Logistic Regression,0.899889,0.787546,0.7525,0.769624,0.956174
1,Tuned KNN,0.907222,0.854103,0.7025,0.770919,0.958837
2,Tuned Gaussian Naive Bayes,0.734444,0.455540,0.9990,0.625744,0.940575


In [36]:
print("Best Logistic Regression:")
print(logistic_grid.best_params_)
print(logistic_grid.best_score_)

print("\nBest KNN:")
print(knn_grid.best_params_)
print(knn_grid.best_score_)

print("\nBest Gaussian Naive Bayes:")
print(nb_grid.best_params_)
print(nb_grid.best_score_)

Best Logistic Regression:
{'model__C': 10, 'model__class_weight': None, 'model__solver': 'liblinear'}
0.9536986830357144

Best KNN:
{'model__metric': 'manhattan', 'model__n_neighbors': 21, 'model__weights': 'distance'}
0.9576446763392857

Best Gaussian Naive Bayes:
{'model__var_smoothing': 1e-12}
0.9359082924107142


In [37]:
import joblib

In [38]:
joblib.dump(
    best_logistic,
    "../models/best_logistic_regression_pipeline.joblib"
)

joblib.dump(
    best_knn,
    "../models/best_knn_pipeline.joblib"
)

joblib.dump(
    best_nb,
    "../models/best_gaussian_nb_pipeline.joblib"
)

['../models/best_gaussian_nb_pipeline.joblib']